# 01 · Data Labeling (Human-in-the-loop)
Giai đoạn 1: auto-label bằng **Qwen2.5-VL-7B-Instruct** (tải từ HuggingFace về local) → duyệt tay bằng Gradio.

**Mọi output (draft JSON, ảnh raw, model) lưu thẳng vào Google Drive** để không mất khi Colab ngắt.

## 1. Mount Google Drive + thiết lập đường dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Gốc dự án trên Drive — tất cả dữ liệu/model nằm ở đây
PROJECT_DRIVE = Path('/content/drive/MyDrive/cccd_project')
RAW_DIR    = PROJECT_DRIVE / 'data' / 'raw'      # ảnh CCCD gốc (tên có 'front'/'back')
DRAFT_DIR  = PROJECT_DRIVE / 'data' / 'draft'    # nhãn draft (JSONL)
MODELS_DIR = PROJECT_DRIVE / 'models'            # model tải về local
QWEN25_LOCAL = MODELS_DIR / 'Qwen2.5-VL-7B-Instruct'
for d in (RAW_DIR, DRAFT_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('RAW   :', RAW_DIR)
print('DRAFT :', DRAFT_DIR)
print('MODEL :', QWEN25_LOCAL)

## 2. Lấy source code + cài thư viện

In [ ]:
!cp -r '/content/drive/MyDrive/cccd_project/code' /content/cccd 2>/dev/null || echo 'Đặt source vào /content/cccd'
%cd /content/cccd
!pip -q install 'transformers>=4.49.0' qwen-vl-utils accelerate Pillow tqdm gradio huggingface_hub

## 3. Tải Qwen2.5-VL-7B-Instruct từ HuggingFace về LOCAL (Drive)
Tải 1 lần, lần sau bỏ qua nếu đã có.

In [ ]:
from huggingface_hub import snapshot_download
if not (QWEN25_LOCAL / 'config.json').exists():
    snapshot_download(repo_id='Qwen/Qwen2.5-VL-7B-Instruct',
                      local_dir=str(QWEN25_LOCAL),
                      ignore_patterns=['*.pth', 'original/*'])
    print('✓ Đã tải về', QWEN25_LOCAL)
else:
    print('✓ Model đã có sẵn local:', QWEN25_LOCAL)

## 4. Auto-label (sinh nhãn draft) — dùng model LOCAL, prompt động theo tên file front/back

In [ ]:
!python -m src.data_pipeline.auto_label \
    --input_dir '{RAW_DIR}' \
    --result_dir '{DRAFT_DIR}' \
    --model_name '{QWEN25_LOCAL}'

## 5. Duyệt & sửa nhãn bằng Gradio (link public share=True)

In [ ]:
DRAFT_FILE = DRAFT_DIR / 'raw_draft.jsonl'
!python -m src.data_pipeline.label_tool \
    --jsonl '{DRAFT_FILE}' \
    --image_root '{PROJECT_DRIVE}' \
    --share